# Joint Angle & Orientation Analysis

Loads saved inference outputs (`.pkl`) and produces an interactive 3D Plotly figure with:
- **Left panel**: dot-product flexion angles (blue = straight, red = bent)
- **Right panel**: relative rotation triads (XYZ axes per joint)

**Prerequisite**: Run `sam3db_pipeline.ipynb` first to generate a `.pkl` file.

## ① Imports

In [ ]:
import sys
sys.path.insert(0, '..')  # make sam3db package importable

import pickle
import numpy as np
from plotly.subplots import make_subplots

from sam3db.joints import ANGLE_TRIPLES, RELATIVE_ROT_PAIRS
from sam3db.visualization import (
    add_skeleton,
    add_joint_angles,
    add_relative_rotation_triads,
)

## ② Load Saved Outputs

In [ ]:
PKL_PATH = '../data/output/your_image_outputs.pkl'  # ← change this

with open(PKL_PATH, 'rb') as f:
    outputs = pickle.load(f)

# Extract joint data for the first detected person
joints   = outputs[0]['pred_joint_coords']  # [127, 3] — metres
glob_rot = outputs[0]['pred_global_rots']   # [127, 3, 3] — rotation matrices

print(f'Loaded {len(outputs)} person(s)')
print(f'joints shape : {joints.shape}')
print(f'glob_rot shape: {glob_rot.shape}')

## ③ Build 3D Figure

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
    subplot_titles=[
        'Joint angles (blue = straight, red = bent)',
        'Relative orientation (R=X, G=Y, B=Z/flex)',
    ],
)

# Left panel — flexion angles
add_skeleton(fig, joints, row=1, col=1)
add_joint_angles(fig, joints, row=1, col=1)

# Right panel — orientation triads
add_skeleton(fig, joints, row=1, col=2)
add_relative_rotation_triads(fig, joints, glob_rot, row=1, col=2)

camera = dict(up=dict(x=0, y=-1, z=0), eye=dict(x=0.5, y=-1.5, z=2))
fig.update_layout(
    title='Joint Analysis — SAM-3D Body',
    width=1600, height=750,
    scene=dict(aspectmode='data', camera=camera),
    scene2=dict(aspectmode='data', camera=camera),
    legend=dict(x=0, xanchor='left', y=1, yanchor='top'),
)

fig.show()

## ④ Save HTML Report

In [ ]:
from pathlib import Path

output_html = f'../data/output/{Path(PKL_PATH).stem}_joint_analysis.html'
fig.write_html(output_html)
print(f' Saved to{output_html}')